### Data Cleaning and Preparation

This notebook cleans FIRMS CSV files from the raw folder and writes the cleaned versions to the processed folder.

Cleaning rules:
- drop all rows with missing values
- for MODIS rows, keep only `confidence > 75` and `frp > 20`
- for VIIRS rows, drop rows where confidence is `l` and keep only `frp > 5`

The code is written to handle future raw FIRMS CSVs too, not only the current Ukraine and Turkey files. Cleaned files use the same name pattern as the raw file, but with `all` replaced by `clnd`.

` The code below sets up the function `

In [2]:
from pathlib import Path
import re

import geopandas as gpd
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
BOUNDARIES_DIR = BASE_DIR.parent / "Boundaries" / "GADM boundaries"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Boundaries folder:", BOUNDARIES_DIR)

COMMON_REQUIRED_COLUMNS = [
    "latitude",
    "longitude",
    "scan",
    "track",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "version",
    "frp",
    "daynight",
    "type",
]

MODIS_REQUIRED_COLUMNS = COMMON_REQUIRED_COLUMNS + ["brightness", "bright_t31"]
VIIRS_REQUIRED_COLUMNS = COMMON_REQUIRED_COLUMNS + ["bright_ti4", "bright_ti5"]
# Ensure cleaned outputs always include these core columns (in this order)
NORMALIZED_OUTPUT_COLUMNS = [
    "latitude",
    "longitude",
    "brightness",
    "scan",
    "track",
    "acq_date",
    "acq_time",
    "satellite",
    "confidence",
    "bright_t31",
    "frp",
    "daynight",
    "country",
    "area_label",
    "bright_ti4",
]

COUNTRY_BOUNDARY_FILES = {
    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
}
COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}


def infer_row_family(frame):
    """Classify each row as MODIS, VIIRS, or UNKNOWN using satellite/instrument text."""
    source_text = (
        frame.get("satellite", pd.Series(index=frame.index, dtype="string")).astype(str).str.lower()
        + " "
        + frame.get("instrument", pd.Series(index=frame.index, dtype="string")).astype(str).str.lower()
    )
    family = pd.Series("unknown", index=frame.index, dtype="string")
    family[source_text.str.contains("modis", na=False)] = "modis"
    family[source_text.str.contains("viirs", na=False)] = "viirs"
    return family


def cleaned_name(raw_path):
    """Turn all_*.csv into clnd_*.csv and keep the rest of the filename stable."""
    stem = raw_path.stem
    if stem.startswith("all_"):
        stem = "clnd_" + stem[len("all_"):]
    elif stem == "all":
        stem = "clnd"
    else:
        stem = "clnd_" + stem
    return raw_path.with_name(stem + raw_path.suffix)


def required_columns_for_family(family_name):
    if family_name == "modis":
        return MODIS_REQUIRED_COLUMNS
    if family_name == "viirs":
        return VIIRS_REQUIRED_COLUMNS
    return COMMON_REQUIRED_COLUMNS


def boundary_file_for_dataset(dataset_name):
    lower_name = dataset_name.lower()
    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
        if country_name in lower_name:
            return country_name, boundary_path
    return None, None


def apply_gadm_cleaning(frame, dataset_name):
    """Keep only points inside country boundary and label area with GADM Admin-1 region names."""
    cleaned = frame.copy()
    if cleaned.empty:
        return cleaned

    cleaned["latitude"] = pd.to_numeric(cleaned.get("latitude"), errors="coerce")
    cleaned["longitude"] = pd.to_numeric(cleaned.get("longitude"), errors="coerce")
    cleaned = cleaned.dropna(subset=["latitude", "longitude"]).copy()
    if cleaned.empty:
        return cleaned

    country_name, boundary_path = boundary_file_for_dataset(dataset_name)
    cleaned["country"] = country_name.title() if country_name else None
    if boundary_path is None or not boundary_path.exists():
        return cleaned

    points_gdf = gpd.GeoDataFrame(
        cleaned,
        geometry=gpd.points_from_xy(cleaned["longitude"], cleaned["latitude"]),
        crs="EPSG:4326",
    )

    country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
    country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)

    # Keep only vertices (points) that are inside/intersecting the country boundary.
    points_gdf = gpd.clip(points_gdf, country_admin0[["geometry"]])
    if points_gdf.empty:
        return cleaned.iloc[0:0].copy()

    region_name_column = "NAME_1"
    if region_name_column not in country_admin1.columns:
        fallback_names = [column for column in country_admin1.columns if column.startswith("NAME")]
        if fallback_names:
            region_name_column = fallback_names[0]
        else:
            region_name_column = None

    if region_name_column is not None:
        admin1_labels = country_admin1[[region_name_column, "geometry"]].rename(
            columns={region_name_column: "gadm_region"}
        )
        points_gdf = gpd.sjoin(points_gdf, admin1_labels, how="left", predicate="intersects")

        existing_area_label = (
            points_gdf["area_label"] if "area_label" in points_gdf.columns else pd.Series(index=points_gdf.index, dtype="string")
        )
        points_gdf["area_label"] = points_gdf["gadm_region"].fillna(existing_area_label)
        points_gdf["area_label"] = points_gdf["area_label"].fillna(country_name.title())

        drop_columns = [column for column in ["gadm_region", "index_right"] if column in points_gdf.columns]
        points_gdf = points_gdf.drop(columns=drop_columns)

    points_gdf = points_gdf.drop(columns=["geometry"])
    return pd.DataFrame(points_gdf)


def clean_firms_frame(frame):
    """Drop missing rows using sensor-specific required fields and apply MODIS/VIIRS quality rules."""
    working = frame.copy()
    family = infer_row_family(working)
    cleaned_frames = []

    for family_name in ["modis", "viirs", "unknown"]:
        family_mask = family.eq(family_name)
        if not family_mask.any():
            continue

        family_rows = working.loc[family_mask].copy()
        required_columns = [column for column in required_columns_for_family(family_name) if column in family_rows.columns]
        family_rows = family_rows.dropna(subset=required_columns, how="any")

        if family_name == "modis":
            modis_conf = pd.to_numeric(family_rows["confidence"], errors="coerce")
            modis_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[(modis_conf > 75) & (modis_frp > 20)]
        elif family_name == "viirs":
            viirs_conf = family_rows["confidence"].astype(str).str.strip().str.lower()
            viirs_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[(viirs_conf != "l") & (viirs_frp > 5)]
        else:
            family_conf = pd.to_numeric(family_rows["confidence"], errors="coerce")
            family_frp = pd.to_numeric(family_rows["frp"], errors="coerce")
            family_rows = family_rows.loc[family_conf.notna() & family_frp.notna()]

        family_rows["cleaning_family"] = family_name
        cleaned_frames.append(family_rows)

    if not cleaned_frames:
        return working.iloc[0:0].copy()

    cleaned = pd.concat(cleaned_frames, ignore_index=True)
    return cleaned


def clean_raw_file(raw_path):
    """Clean one raw FIRMS CSV and write the result to the processed folder."""
    frame = pd.read_csv(raw_path)
    cleaned = clean_firms_frame(frame)
    cleaned = apply_gadm_cleaning(cleaned, raw_path.stem)
    available_output_columns = [column for column in NORMALIZED_OUTPUT_COLUMNS if column in cleaned.columns]
    cleaned = cleaned.loc[:, available_output_columns].copy()

    output_path = PROCESSED_DIR / cleaned_name(raw_path).name
    cleaned.to_csv(output_path, index=False)
    return output_path, len(frame), len(cleaned)

Raw folder: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\raw
Processed folder: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed
Boundaries folder: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\Boundaries\GADM boundaries


` Execute the functions for data in 'raw' folder `

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv"))

if not raw_files:
    print("No CSV files found in:", RAW_DIR)
else:
    results = []
    for raw_file in raw_files:
        output_path, before_rows, after_rows = clean_raw_file(raw_file)
        results.append({
            "raw_file": raw_file.name,
            "processed_file": output_path.name,
            "rows_before": before_rows,
            "rows_after": after_rows,
        })
        print(f"{raw_file.name} -> {output_path.name} | {before_rows} rows -> {after_rows} rows")

    summary = pd.DataFrame(results)
    print("\nCleaning summary:")
    display(summary)

C:\Users\AFT\AppData\Local\Temp\ipykernel_36828\1388011300.py:206: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  frame = pd.read_csv(raw_path)


---
### Mapping after cleaning

In [ ]:
# Generate an interactive map for each cleaned (`clnd_*.csv`) dataset using the matching GADM boundaries.
#from pathlib import Path
#
#import geopandas as gpd
#import pandas as pd
#import folium
#from folium.plugins import HeatMap
#
#OUTPUT_DIR = BASE_DIR / "outputs" / "maps"
#BOUNDARIES_DIR = BASE_DIR.parent / "Boundaries" / "GADM boundaries"
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
#COUNTRY_BOUNDARY_FILES = {
#    "ukraine":      BOUNDARIES_DIR / "gadm41_UKR.gpkg",
#    "turkey":       BOUNDARIES_DIR / "gadm41_TUR.gpkg",
#    "iran":         BOUNDARIES_DIR / "gadm41_IRN.gpkg",
#    "israel":       BOUNDARIES_DIR / "gadm41_ISR.gpkg",
#    "qatar":        BOUNDARIES_DIR / "gadm41_QAT.gpkg",
#    "uae":          BOUNDARIES_DIR / "gadm41_ARE.gpkg",
#    "australia":    BOUNDARIES_DIR / "gadm41_AUS.gpkg",
#}
#COUNTRY_BOUNDARY_FILES = {k: v for k, v in COUNTRY_BOUNDARY_FILES.items() if v.exists()}
#
#
#def boundary_file_for_dataset(dataset_name):
#    lower_name = dataset_name.lower()
#    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
#        if country_name in lower_name:
#            return country_name, boundary_path
#    return None, None
#
#
#processed_files = sorted(PROCESSED_DIR.glob("clnd_*.csv"))
#if not processed_files:
#    print("No cleaned files found in:", PROCESSED_DIR)
#else:
#    for pf in processed_files:
#        country_name, boundary_path = boundary_file_for_dataset(pf.stem)
#        if boundary_path is None or not boundary_path.exists():
#            print(f"{pf.name}: no matching GADM boundary file found, skipping")
#            continue
#
#        print("Processing:", pf.name)
#        print("Using boundary:", boundary_path.name)
#
#        df = pd.read_csv(pf)
#        df["latitude"] = pd.to_numeric(df.get("latitude"), errors="coerce")
#        df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
#        df = df.dropna(subset=["latitude", "longitude"]).copy()
#        if df.empty:
#            print(f"{pf.name}: no valid points, skipping")
#            continue
#
#        points_gdf = gpd.GeoDataFrame(
#            df,
#            geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
#            crs="EPSG:4326",
#        )
#
#        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
#        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
#
#        country_points = gpd.clip(points_gdf, country_admin0[["geometry"]])
#        country_points = country_points.dropna(subset=["geometry"]).copy()
#        if country_points.empty:
#            print(f"{pf.name}: no points inside {country_name}, skipping")
#            continue
#
#        center_lat = country_points.geometry.y.median()
#        center_lon = country_points.geometry.x.median()
#        m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")
#
#        folium.GeoJson(
#            country_admin0.to_json(),
#            name=f"{country_name.title()} boundary",
#            style_function=lambda _: {"color": "#111827", "weight": 2.2, "fillOpacity": 0},
#        ).add_to(m)
#
#        folium.GeoJson(
#            country_admin1.to_json(),
#            name="Admin-1 boundaries",
#            style_function=lambda _: {"color": "#6b7280", "weight": 0.8, "fillOpacity": 0},
#        ).add_to(m)
#
#        heat_data = []
#        for _, row in country_points.iterrows():
#            w = row.get("brightness")
#            if pd.isna(w):
#                w = row.get("frp")
#            try:
#                weight = float(w) if pd.notna(w) else 1.0
#            except Exception:
#                weight = 1.0
#            heat_data.append([row["latitude"], row["longitude"], weight])
#
#        if heat_data:
#            HeatMap(heat_data, radius=12, blur=10, min_opacity=0.3, name="Fire intensity").add_to(m)
#
#        for _, row in country_points.iterrows():
#            daynight = row.get("daynight", "")
#            color = "#ef4444" if daynight == "D" else "#f59e0b"
#
#            # Marker size is scaled using FRP only.
#            frp_value = row.get("frp")
#            try:
#                frp_value = float(frp_value) if pd.notna(frp_value) else 0.0
#            except Exception:
#                frp_value = 0.0
#            radius = 3 + min(frp_value / 10.0, 9)
#
#            popup_text = (
#                f"Satellite: {row.get('satellite', '')}<br>"
#                f"Date: {row.get('acq_date', '')} {row.get('acq_time', '')}<br>"
#                f"FRP: {row.get('frp', '')}<br>"
#                f"Confidence: {row.get('confidence', '')}<br>"
#                f"Area: {row.get('area_label', '')}"
#            )
#            folium.CircleMarker(
#                location=[row["latitude"], row["longitude"]],
#                radius=radius,
#                color=color,
#                weight=1,
#                fill=True,
#                fill_color=color,
#                fill_opacity=0.75,
#                popup=folium.Popup(popup_text, max_width=300),
#            ).add_to(m)
#
#        folium.LayerControl(collapsed=False).add_to(m)
#        outpath = OUTPUT_DIR / (pf.stem + ".html")
#        m.save(str(outpath))
#        print(f"Saved map: {outpath}")

---
### Clustering

In [4]:
# Cluster cleaned FIRMS points using time and distance thresholds, then summarize each cluster.
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
from IPython.display import display

TIME_WINDOW_HOURS = 6
DISTANCE_THRESHOLDS_M = {
    "modis": 2000.0,
    "viirs": 800.0,
}

CLUSTER_OUTPUT_COLUMNS = [
    "sensor",
    "n_points",
    "centroid_lat",
    "centroid_lon",
    "max_brightness",
    "maxbrightness_ti4",
    "average_confidence_rate",
    "sum_frp",
    "max_frp",
    "earliest_detection",
    "latest_detection",
    "duration_hours",
    "centroid_country",
    "centroid_area_label",
]


def infer_sensor_family_from_satellite(series):
    sat = series.astype(str).str.lower()
    family = pd.Series("unknown", index=series.index, dtype="string")
    family[sat.str.contains("modis", na=False)] = "modis"
    family[sat.str.contains("viirs", na=False)] = "viirs"
    return family


def parse_detection_datetime(frame):
    date_part = pd.to_datetime(frame.get("acq_date"), errors="coerce")
    time_raw = frame.get("acq_time", pd.Series(index=frame.index, dtype="object")).astype(str)
    time_digits = time_raw.str.extract(r"(\d+)", expand=False).fillna("0").str.zfill(4).str[-4:]

    hours = pd.to_numeric(time_digits.str[:2], errors="coerce").fillna(0).clip(lower=0, upper=23)
    minutes = pd.to_numeric(time_digits.str[2:], errors="coerce").fillna(0).clip(lower=0, upper=59)
    return date_part + pd.to_timedelta(hours * 60 + minutes, unit="m")


def boundary_file_for_dataset_name(dataset_name):
    lower_name = dataset_name.lower()
    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
        if country_name in lower_name:
            return country_name, boundary_path
    return None, None


def clustered_name_from_cleaned(cleaned_name):
    stem = Path(cleaned_name).stem
    if stem.startswith("clnd_"):
        return "clstrd_" + stem[len("clnd_"):] + ".csv"
    if stem == "clnd":
        return "clstrd.csv"
    return "clstrd_" + stem + ".csv"


def union_find_labels(geometries, times, eps_m, max_hours):
    n = len(geometries)
    if n == 0:
        return np.array([], dtype=int)

    parent = np.arange(n)
    rank = np.zeros(n, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    sindex = geometries.sindex

    for i, geom_i in enumerate(geometries):
        bounds = geom_i.buffer(eps_m).bounds
        candidates = list(sindex.intersection(bounds))
        time_i = times[i]
        for j in candidates:
            if j <= i:
                continue
            delta_hours = abs((times[j] - time_i).total_seconds()) / 3600.0
            if delta_hours > max_hours:
                continue
            if geom_i.distance(geometries.iloc[j]) <= eps_m:
                union(i, j)

    roots = np.array([find(i) for i in range(n)])
    root_to_label = {}
    labels = np.zeros(n, dtype=int)
    next_label = 0
    for idx, root in enumerate(roots):
        if root not in root_to_label:
            root_to_label[root] = next_label
            next_label += 1
        labels[idx] = root_to_label[root]
    return labels


def keep_consecutive_time_runs(frame, label_col, time_col, max_hours):
    """Keep only runs where consecutive detections are <= max_hours; singleton runs are dropped."""
    kept_runs = []
    next_cluster_id = 0

    for _, grp in frame.groupby(label_col, sort=False):
        grp = grp.sort_values(time_col).copy()
        idx_list = grp.index.tolist()
        if not idx_list:
            continue

        run = [idx_list[0]]
        for curr_idx in idx_list[1:]:
            prev_idx = run[-1]
            gap_h = (grp.loc[curr_idx, time_col] - grp.loc[prev_idx, time_col]).total_seconds() / 3600.0
            if gap_h <= max_hours:
                run.append(curr_idx)
            else:
                if len(run) >= 2:
                    run_df = grp.loc[run].copy()
                    run_df[label_col] = next_cluster_id
                    kept_runs.append(run_df)
                    next_cluster_id += 1
                run = [curr_idx]

        if len(run) >= 2:
            run_df = grp.loc[run].copy()
            run_df[label_col] = next_cluster_id
            kept_runs.append(run_df)
            next_cluster_id += 1

    if not kept_runs:
        return frame.iloc[0:0].copy()
    return pd.concat(kept_runs, ignore_index=True)


def assign_centroid_area(cluster_frame, country_admin1, country_name):
    centroids_gdf = gpd.GeoDataFrame(
        cluster_frame.copy(),
        geometry=gpd.points_from_xy(cluster_frame["centroid_lon"], cluster_frame["centroid_lat"]),
        crs="EPSG:4326",
    )
    cluster_frame["centroid_country"] = country_name.title()

    region_name_column = "NAME_1"
    if region_name_column not in country_admin1.columns:
        name_candidates = [column for column in country_admin1.columns if column.startswith("NAME")]
        region_name_column = name_candidates[0] if name_candidates else None

    if region_name_column is None:
        cluster_frame["centroid_area_label"] = country_name.title()
        return cluster_frame

    labels = country_admin1[[region_name_column, "geometry"]].rename(columns={region_name_column: "centroid_area_label"})
    joined = gpd.sjoin(centroids_gdf, labels, how="left", predicate="intersects")
    cluster_frame["centroid_area_label"] = joined["centroid_area_label"].fillna(country_name.title())
    return cluster_frame


processed_files = sorted(PROCESSED_DIR.glob("clnd_*.csv"))
if not processed_files:
    print("No cleaned files found in:", PROCESSED_DIR)
else:
    for pf in processed_files:
        dataset_name = pf.stem
        print("\nClustering:", pf.name)

        country_name, boundary_path = boundary_file_for_dataset_name(dataset_name)
        if boundary_path is None or not boundary_path.exists():
            print(f"{pf.name}: no matching GADM boundary file found, skipping")
            continue

        df = pd.read_csv(pf)
        if df.empty:
            print(f"{pf.name}: empty dataset, skipping")
            continue

        df["latitude"] = pd.to_numeric(df.get("latitude"), errors="coerce")
        df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
        df["brightness"] = pd.to_numeric(df.get("brightness"), errors="coerce")
        df["frp"] = pd.to_numeric(df.get("frp"), errors="coerce")
        df["detected_at"] = parse_detection_datetime(df)
        df = df.dropna(subset=["latitude", "longitude", "detected_at"]).copy()
        if df.empty:
            print(f"{pf.name}: no valid rows after datetime/coordinate parsing")
            continue

        # Ensure only in-boundary points are clustered.
        points_gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
            crs="EPSG:4326",
        )
        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
        points_gdf = gpd.clip(points_gdf, country_admin0[["geometry"]])
        points_gdf = points_gdf.dropna(subset=["geometry"]).copy()
        if points_gdf.empty:
            print(f"{pf.name}: no points remain inside boundary")
            continue

        points_gdf["sensor_family"] = infer_sensor_family_from_satellite(points_gdf["satellite"])
        dataset_clusters = []

        for family_name, eps_m in DISTANCE_THRESHOLDS_M.items():
            family_points = points_gdf.loc[points_gdf["sensor_family"] == family_name].copy()
            if family_points.empty:
                continue

            projected_crs = family_points.estimate_utm_crs()
            family_proj = family_points.to_crs(projected_crs)
            family_proj = family_proj.sort_values("detected_at").reset_index(drop=True)

            labels = union_find_labels(
                family_proj.geometry,
                family_proj["detected_at"].tolist(),
                eps_m=eps_m,
                max_hours=TIME_WINDOW_HOURS,
            )

            family_proj["cluster_local_id"] = labels
            family_geo = family_proj.to_crs(epsg=4326)

            # Keep only consecutive (<=6h) runs and drop points with >6h gaps from runs.
            family_geo = keep_consecutive_time_runs(
                family_geo,
                label_col="cluster_local_id",
                time_col="detected_at",
                max_hours=TIME_WINDOW_HOURS,
            )
            if family_geo.empty:
                continue

            # Aggregate per cluster.
            for cluster_id, grp in family_geo.groupby("cluster_local_id", sort=True):
                earliest = grp["detected_at"].min()
                latest = grp["detected_at"].max()
                duration_h = (latest - earliest).total_seconds() / 3600.0
                cluster_geom = gpd.GeoSeries(grp.geometry, crs="EPSG:4326").union_all().centroid

                # Compute average confidence rate
                if family_name == "modis":
                    avg_conf = grp["confidence"].dropna().astype(float).mean()
                    avg_conf = round(avg_conf, 1) if pd.notna(avg_conf) else np.nan
                else:
                    conf_series = grp["confidence"].dropna().astype(str).str.strip().str.lower()
                    avg_conf = conf_series.mode()[0] if not conf_series.empty else "n"

                # Compute maxbrightness_ti4
                if "bright_ti4" in grp.columns and grp["bright_ti4"].notna().any():
                    max_ti4 = float(grp["bright_ti4"].max())
                else:
                    max_ti4 = np.nan

                dataset_clusters.append(
                    {
                        "sensor": family_name,
                        "n_points": int(len(grp)),
                        "centroid_lat": float(cluster_geom.y),
                        "centroid_lon": float(cluster_geom.x),
                        "max_brightness": float(grp["brightness"].max()) if grp["brightness"].notna().any() else np.nan,
                        "maxbrightness_ti4": max_ti4,
                        "average_confidence_rate": avg_conf,
                        "sum_frp": float(grp["frp"].sum(skipna=True)),
                        "max_frp": float(grp["frp"].max()) if grp["frp"].notna().any() else np.nan,
                        "earliest_detection": earliest,
                        "latest_detection": latest,
                        "duration_hours": duration_h,
                    }
                )

        if not dataset_clusters:
            print(f"{pf.name}: no clusters produced")
            continue

        cluster_table = pd.DataFrame(dataset_clusters)
        cluster_table = assign_centroid_area(cluster_table, country_admin1, country_name)
        cluster_table = cluster_table.loc[:, CLUSTER_OUTPUT_COLUMNS].copy()
        cluster_table = cluster_table.sort_values(["sensor", "earliest_detection", "n_points"], ascending=[True, True, False])

        out_file = PROCESSED_DIR / clustered_name_from_cleaned(pf.name)
        cluster_table.to_csv(out_file, index=False)
        print(f"Saved cluster file: {out_file} | clusters: {len(cluster_table)}")
        display(cluster_table.head(10))


Clustering: clnd_20210701_90d_turkey.csv
Saved cluster file: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed\clstrd_20210701_90d_turkey.csv | clusters: 1634


,sensor,n_points,centroid_lat,centroid_lon,max_brightness,maxbrightness_ti4,average_confidence_rate,sum_frp,max_frp,earliest_detection,latest_detection,duration_hours,centroid_country,centroid_area_label
0,modis,2,37.124800,40.063550,346.8,NaN,87.0,179.3,99.3,2021-07-01 07:41:00,2021-07-01 07:41:00,0.000000,Turkey,Sanliurfa
1,modis,3,37.478500,38.665800,322.6,NaN,97.0,79.7,31.6,2021-07-07 19:46:00,2021-07-07 19:46:00,0.000000,Turkey,Sanliurfa
2,modis,2,37.650950,42.658850,352.9,NaN,90.5,149.8,93.5,2021-07-12 10:41:00,2021-07-12 10:41:00,0.000000,Turkey,Sirnak
3,modis,2,39.564000,27.795650,331.8,NaN,79.5,42.1,21.1,2021-07-14 08:48:00,2021-07-14 08:48:00,0.000000,Turkey,Balikesir
4,modis,3,37.812933,38.513233,315.4,NaN,88.3,63.1,22.2,2021-07-14 19:52:00,2021-07-14 19:52:00,0.000000,Turkey,Adiyaman
5,modis,23,36.216583,33.230000,397.7,NaN,93.7,5757.7,1226.6,2021-07-15 07:53:00,2021-07-15 11:12:00,3.316667,Turkey,Mersin
6,modis,2,36.860250,39.852600,345.6,NaN,86.0,83.4,43.3,2021-07-15 07:53:00,2021-07-15 07:53:00,0.000000,Turkey,Sanliurfa
7,modis,2,37.045100,36.175350,346.5,NaN,90.0,186.5,96.1,2021-07-15 11:12:00,2021-07-15 11:12:00,0.000000,Turkey,Osmaniye
8,modis,2,37.359900,37.194000,340.9,NaN,82.5,126.4,77.9,2021-07-15 11:12:00,2021-07-15 11:12:00,0.000000,Turkey,K. Maras
9,modis,5,36.877180,36.496780,328.5,NaN,97.6,468.6,123.9,2021-07-15 18:56:00,2021-07-15 23:14:00,4.300000,Turkey,Hatay



Clustering: clnd_20220401_1100d_ukraine.csv
Saved cluster file: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed\clstrd_20220401_1100d_ukraine.csv | clusters: 28040


,sensor,n_points,centroid_lat,centroid_lon,max_brightness,maxbrightness_ti4,average_confidence_rate,sum_frp,max_frp,earliest_detection,latest_detection,duration_hours,centroid_country,centroid_area_label
0,modis,2,47.14695,35.39725,338.0,NaN,83.0,81.4,59.7,2022-04-01 08:58:00,2022-04-01 08:58:00,0.0,Ukraine,Zaporizhia
1,modis,2,49.17295,33.50590,332.7,NaN,82.0,159.6,108.0,2022-04-02 08:02:00,2022-04-02 08:02:00,0.0,Ukraine,Poltava
2,modis,2,49.10755,39.41245,353.5,NaN,93.0,103.3,67.4,2022-04-07 08:20:00,2022-04-07 08:20:00,0.0,Ukraine,Luhans'k
3,modis,2,47.89070,37.62320,365.9,NaN,96.5,146.5,99.6,2022-04-07 08:21:00,2022-04-07 08:21:00,0.0,Ukraine,Donets'k
4,modis,2,47.90800,39.10660,327.6,NaN,81.5,147.1,83.4,2022-04-08 09:03:00,2022-04-08 09:03:00,0.0,Ukraine,Luhans'k
5,modis,2,47.91570,39.07540,357.3,NaN,97.0,497.3,289.4,2022-04-08 09:03:00,2022-04-08 09:03:00,0.0,Ukraine,Luhans'k
6,modis,2,48.41790,38.07835,336.2,NaN,88.0,192.0,104.3,2022-04-08 09:03:00,2022-04-08 09:03:00,0.0,Ukraine,Donets'k
7,modis,2,48.72535,25.23125,324.8,NaN,80.0,65.5,38.6,2022-04-09 09:47:00,2022-04-09 09:47:00,0.0,Ukraine,Ivano-Frankivs'k
8,modis,2,49.29575,38.38105,330.9,NaN,84.0,97.9,51.6,2022-04-11 07:56:00,2022-04-11 07:56:00,0.0,Ukraine,Luhans'k
9,modis,2,48.90575,27.78475,352.8,NaN,89.0,108.8,84.5,2022-04-13 09:22:00,2022-04-13 09:22:00,0.0,Ukraine,Vinnytsya



Clustering: clnd_20250601_90d_turkey.csv
Saved cluster file: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed\clstrd_20250601_90d_turkey.csv | clusters: 1466


,sensor,n_points,centroid_lat,centroid_lon,max_brightness,maxbrightness_ti4,average_confidence_rate,sum_frp,max_frp,earliest_detection,latest_detection,duration_hours,centroid_country,centroid_area_label
0,modis,2,36.901350,35.483400,330.9,NaN,79.5,207.1,121.0,2025-06-04 11:12:00,2025-06-04 11:12:00,0.0,Turkey,Adana
1,modis,3,38.290167,34.008733,338.1,NaN,86.7,145.5,57.9,2025-06-05 07:54:00,2025-06-05 07:54:00,0.0,Turkey,Aksaray
2,modis,3,37.204700,33.320700,337.4,NaN,87.3,130.1,47.8,2025-06-05 07:54:00,2025-06-05 07:54:00,0.0,Turkey,Karaman
3,modis,2,36.554400,36.506100,420.5,NaN,92.5,404.6,377.9,2025-06-05 11:50:00,2025-06-05 11:50:00,0.0,Turkey,Hatay
5,modis,4,38.879200,27.130225,380.1,NaN,92.8,271.0,147.0,2025-06-10 07:55:00,2025-06-10 07:55:00,0.0,Turkey,Izmir
4,modis,2,38.284950,34.001750,339.7,NaN,79.5,78.1,51.2,2025-06-10 07:55:00,2025-06-10 11:49:00,3.9,Turkey,Aksaray
8,modis,3,39.687967,30.728533,333.7,NaN,82.7,88.4,34.0,2025-06-10 11:49:00,2025-06-10 11:49:00,0.0,Turkey,Eskisehir
6,modis,2,36.831750,34.963050,384.7,NaN,90.0,206.6,183.6,2025-06-10 11:49:00,2025-06-10 11:49:00,0.0,Turkey,Mersin
7,modis,2,37.005500,35.861300,353.1,NaN,90.5,91.0,60.5,2025-06-10 11:49:00,2025-06-10 11:49:00,0.0,Turkey,Adana
9,modis,2,37.564850,39.427300,317.7,NaN,82.5,54.4,28.1,2025-06-12 18:38:00,2025-06-12 18:38:00,0.0,Turkey,Sanliurfa



Clustering: clnd_20250625_35d_turkey.csv
Saved cluster file: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed\clstrd_20250625_35d_turkey.csv | clusters: 721


,sensor,n_points,centroid_lat,centroid_lon,max_brightness,maxbrightness_ti4,average_confidence_rate,sum_frp,max_frp,earliest_detection,latest_detection,duration_hours,centroid_country,centroid_area_label
0,modis,5,36.755240,39.584560,368.0,NaN,91.6,1238.5,456.7,2025-06-25 07:57:00,2025-06-25 11:42:00,3.750000,Turkey,Sanliurfa
1,modis,2,36.943150,39.567550,340.6,NaN,86.5,384.3,214.8,2025-06-25 07:57:00,2025-06-25 07:57:00,0.000000,Turkey,Sanliurfa
2,modis,2,37.872250,32.561150,336.1,NaN,82.0,78.0,42.1,2025-06-25 11:42:00,2025-06-25 11:42:00,0.000000,Turkey,Konya
3,modis,10,38.702250,26.932250,384.8,NaN,95.9,2368.0,1045.5,2025-06-25 11:43:00,2025-06-25 19:00:00,7.283333,Turkey,Izmir
4,modis,2,37.164700,39.794150,317.8,NaN,90.5,140.3,80.3,2025-06-25 18:59:00,2025-06-25 18:59:00,0.000000,Turkey,Sanliurfa
5,modis,2,37.257250,40.065750,347.0,NaN,84.5,64.8,39.0,2025-06-26 06:59:00,2025-06-26 06:59:00,0.000000,Turkey,Mardin
6,modis,3,36.934433,39.553333,331.9,NaN,100.0,93.7,41.9,2025-06-26 18:00:00,2025-06-26 18:00:00,0.000000,Turkey,Sanliurfa
7,modis,4,37.321450,39.699250,339.1,NaN,100.0,215.6,58.2,2025-06-26 18:01:00,2025-06-26 18:01:00,0.000000,Turkey,Sanliurfa
8,modis,9,38.707244,26.956678,354.5,NaN,99.8,1025.6,220.6,2025-06-26 19:39:00,2025-06-26 19:39:00,0.000000,Turkey,Izmir
10,modis,4,38.579775,27.095225,384.8,NaN,100.0,1540.7,640.0,2025-06-27 01:58:00,2025-06-27 01:58:00,0.000000,Turkey,Izmir



Clustering: clnd_20260101_120d_iran.csv
Saved cluster file: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\FIRMS\data\processed\clstrd_20260101_120d_iran.csv | clusters: 3726


,sensor,n_points,centroid_lat,centroid_lon,max_brightness,maxbrightness_ti4,average_confidence_rate,sum_frp,max_frp,earliest_detection,latest_detection,duration_hours,centroid_country,centroid_area_label
1,modis,3,31.086867,47.791167,334.2,NaN,84.3,149.5,77.4,2026-01-01 06:29:00,2026-01-01 11:10:00,4.683333,Iran,Khuzestan
0,modis,2,32.348600,47.704100,322.8,NaN,81.5,71.3,39.4,2026-01-01 06:29:00,2026-01-01 11:11:00,4.700000,Iran,Ilam
2,modis,2,30.716900,49.825000,327.1,NaN,83.5,96.8,57.9,2026-01-01 06:29:00,2026-01-01 11:10:00,4.683333,Iran,Khuzestan
3,modis,2,30.212750,50.786050,316.5,NaN,77.0,41.5,20.9,2026-01-01 11:10:00,2026-01-01 11:10:00,0.000000,Iran,Kohgiluyeh and Buyer Ahmad
4,modis,2,30.723000,49.820600,337.7,NaN,100.0,71.8,43.1,2026-01-01 17:28:00,2026-01-01 17:28:00,0.000000,Iran,Khuzestan
5,modis,2,31.278500,48.121950,319.3,NaN,92.5,84.7,48.2,2026-01-02 23:51:00,2026-01-02 23:51:00,0.000000,Iran,Khuzestan
6,modis,3,31.271633,48.115533,319.0,NaN,92.0,125.6,48.8,2026-01-03 17:08:00,2026-01-03 17:08:00,0.000000,Iran,Khuzestan
7,modis,2,31.092100,47.795650,324.2,NaN,95.0,63.9,38.1,2026-01-04 00:29:00,2026-01-04 00:29:00,0.000000,Iran,Khuzestan
8,modis,3,31.094233,47.788867,323.3,NaN,96.7,94.9,35.3,2026-01-04 17:47:00,2026-01-04 17:47:00,0.000000,Iran,Khuzestan
9,modis,2,31.273850,48.116900,325.4,NaN,100.0,73.6,40.5,2026-01-04 17:47:00,2026-01-04 17:47:00,0.000000,Iran,Khuzestan


`Clustered Map`

In [ ]:
# Generate an interactive map for each clustered (`clstrd_*.csv`) dataset using matching GADM boundaries.
#import folium
#import pandas as pd
#from pathlib import Path
#import geopandas as gpd
#
#CLUSTER_MAP_DIR = BASE_DIR / "outputs" / "maps"
#CLUSTER_MAP_DIR.mkdir(parents=True, exist_ok=True)
#
#
#def boundary_file_for_clustered_dataset(dataset_name):
#    lower_name = dataset_name.lower()
#    for country_name, boundary_path in COUNTRY_BOUNDARY_FILES.items():
#        if country_name in lower_name:
#            return country_name, boundary_path
#    return None, None
#
#
#clustered_files = sorted(PROCESSED_DIR.glob("clstrd_*.csv"))
#if not clustered_files:
#    print("No clustered files found in:", PROCESSED_DIR)
#else:
#    for cf in clustered_files:
#        dataset_name = cf.stem
#        country_name, boundary_path = boundary_file_for_clustered_dataset(dataset_name)
#        if boundary_path is None or not boundary_path.exists():
#            print(f"{cf.name}: no matching GADM boundary file found, skipping")
#            continue
#
#        print("Processing clustered file:", cf.name)
#        print("Using boundary:", boundary_path.name)
#
#        cluster_df = pd.read_csv(cf)
#        cluster_df["centroid_lat"] = pd.to_numeric(cluster_df.get("centroid_lat"), errors="coerce")
#        cluster_df["centroid_lon"] = pd.to_numeric(cluster_df.get("centroid_lon"), errors="coerce")
#        cluster_df["sum_frp"] = pd.to_numeric(cluster_df.get("sum_frp"), errors="coerce")
#        cluster_df["max_frp"] = pd.to_numeric(cluster_df.get("max_frp"), errors="coerce")
#        cluster_df = cluster_df.dropna(subset=["centroid_lat", "centroid_lon"]).copy()
#        if cluster_df.empty:
#            print(f"{cf.name}: no valid centroid rows, skipping")
#            continue
#
#        country_admin0 = gpd.read_file(boundary_path, layer="ADM_ADM_0").to_crs(epsg=4326)
#        country_admin1 = gpd.read_file(boundary_path, layer="ADM_ADM_1").to_crs(epsg=4326)
#
#        center_lat = cluster_df["centroid_lat"].median()
#        center_lon = cluster_df["centroid_lon"].median()
#        m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")
#
#        folium.GeoJson(
#            country_admin0.to_json(),
#            name=f"{country_name.title()} boundary",
#            style_function=lambda _: {"color": "#111827", "weight": 2.2, "fillOpacity": 0},
#        ).add_to(m)
#
#        folium.GeoJson(
#            country_admin1.to_json(),
#            name="Admin-1 boundaries",
#            style_function=lambda _: {"color": "#6b7280", "weight": 0.8, "fillOpacity": 0},
#        ).add_to(m)
#
#        for _, row in cluster_df.iterrows():
#            sensor = str(row.get("sensor", "")).lower()
#            color = "#2563eb" if sensor == "modis" else "#ef4444"
#
#            sum_frp_val = row.get("sum_frp")
#            if pd.isna(sum_frp_val):
#                sum_frp_val = row.get("max_frp")
#            try:
#                sum_frp_val = float(sum_frp_val) if pd.notna(sum_frp_val) else 0.0
#            except Exception:
#                sum_frp_val = 0.0
#
#            radius = 4 + min(sum_frp_val / 20.0, 10)
#
#            popup_text = (
#                f"Sensor: {row.get('sensor', '')}<br>"
#                f"Points: {row.get('n_points', '')}<br>"
#                f"Max Brightness: {row.get('max_brightness', '')}<br>"
#                f"Maxbrightness TI4: {row.get('maxbrightness_ti4', '')}<br>"
#                f"Average Confidence: {row.get('average_confidence_rate', '')}<br>"
#                f"Sum FRP: {row.get('sum_frp', '')}<br>"
#                f"Max FRP: {row.get('max_frp', '')}<br>"
#                f"Earliest: {row.get('earliest_detection', '')}<br>"
#                f"Latest: {row.get('latest_detection', '')}<br>"
#                f"Duration (h): {row.get('duration_hours', '')}<br>"
#                f"Centroid Country: {row.get('centroid_country', '')}<br>"
#                f"Centroid Area: {row.get('centroid_area_label', '')}"
#            )
#
#            folium.CircleMarker(
#                location=[row["centroid_lat"], row["centroid_lon"]],
#                radius=radius,
#                color=color,
#                weight=1,
#                fill=True,
#                fill_color=color,
#                fill_opacity=0.75,
#                popup=folium.Popup(popup_text, max_width=320),
#            ).add_to(m)
#
#        folium.LayerControl(collapsed=False).add_to(m)
#
#        html_name = dataset_name + "_map.html"
#        outpath = CLUSTER_MAP_DIR / html_name
#        m.save(str(outpath))
#        print(f"Saved clustered map: {outpath}")